In [1]:
# ============================================================
# Autosave helpers for Quad4FHE plaintext experiments
# - duplicates notebook stdout/stderr to a .txt log
# - writes structured JSON after each quad4fhe.replace(...) run
# - writes a compact CSV summary
# ============================================================
from __future__ import annotations

import contextlib
import json
import math
import re
import sys
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


_AUTOSAVE_PAYLOAD: Optional[Dict[str, Any]] = None
_AUTOSAVE_RUNS = []
_AUTOSAVE_SUMMARY_ROWS = []


class _AutosaveTeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


@contextlib.contextmanager
def tee_stdout_stderr(log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old_stdout, old_stderr = sys.stdout, sys.stderr
    with log_path.open("w", encoding="utf-8", buffering=1) as fh:
        sys.stdout = _AutosaveTeeStream(old_stdout, fh)
        sys.stderr = _AutosaveTeeStream(old_stderr, fh)
        try:
            print(f"[autosave] stdout/stderr log -> {log_path}")
            yield log_path
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _to_jsonable(obj: Any) -> Any:
    try:
        import numpy as _np
    except Exception:
        _np = None
    try:
        import pandas as _pd
    except Exception:
        _pd = None
    try:
        import torch as _torch
    except Exception:
        _torch = None

    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if _np is not None:
        if isinstance(obj, _np.integer):
            return int(obj)
        if isinstance(obj, _np.floating):
            value = float(obj)
            return value if math.isfinite(value) else None
        if isinstance(obj, _np.bool_):
            return bool(obj)
        if isinstance(obj, _np.ndarray):
            return _to_jsonable(obj.tolist())
    if _torch is not None and hasattr(_torch, "is_tensor") and _torch.is_tensor(obj):
        return _to_jsonable(obj.detach().cpu().numpy())
    if isinstance(obj, Path):
        return str(obj)
    if _pd is not None:
        if isinstance(obj, _pd.DataFrame):
            return _to_jsonable(obj.to_dict(orient="records"))
        if isinstance(obj, _pd.Series):
            return _to_jsonable(obj.to_dict())
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(v) for v in obj]
    return str(obj)


def _save_json(obj: Dict[str, Any], path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(_to_jsonable(obj), fh, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"[autosave] JSON -> {path}")
    return path


def _save_csv(rows: Iterable[Dict[str, Any]], path) -> Path:
    import pandas as _pd
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _pd.DataFrame(list(rows)).to_csv(path, index=False)
    print(f"[autosave] CSV -> {path}")
    return path


def _dataframe_records(df: Any) -> list:
    if df is None:
        return []
    try:
        return _to_jsonable(df.to_dict(orient="records"))
    except Exception:
        return []


def _dataframe_test_by_method(df: Any) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    if df is None:
        return out
    try:
        sub = df[df["Split"] == "test"]
        for _, row in sub.iterrows():
            method = str(row.get("Method"))
            metrics = {str(k): _to_jsonable(v) for k, v in row.to_dict().items()
                       if k not in ("Method", "Split")}
            out[method] = metrics
    except Exception:
        return out
    return out


def _metric_from_table(table: Dict[str, Dict[str, Any]], method: str, *keys: str) -> Any:
    row = table.get(method, {})
    for key in keys:
        if key in row:
            return row[key]
    return None


def _quad_report_diagnostics(result: Any, split: str, n_expected: Optional[int] = None) -> Dict[str, Any]:
    if split in ("fit", "calib", "calibration"):
        attr_candidates = ["fit_diagnostics", "calib_diagnostics", "calibration_diagnostics"]
        split_label = "fit"
    else:
        attr_candidates = ["test_diagnostics"]
        split_label = "test"

    diag = None
    for attr in attr_candidates:
        value = getattr(result, attr, None)
        if value is not None:
            diag = dict(value)
            break

    if diag is None:
        diag = {}
        df = getattr(result, "report_vs_pseudo", None)
        if df is not None:
            try:
                row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split_label)]
                if len(row) > 0:
                    diag["agreement"] = float(row.iloc[0]["Agreement"])
            except Exception:
                pass

    n_value = diag.get("n", diag.get("calib_n", diag.get("n_samples", n_expected)))
    agreement = diag.get("agreement", diag.get("calib_agreement", diag.get("fit_agreement")))
    mismatch = diag.get("mismatch_count", diag.get("calib_mismatch_count", diag.get("fit_mismatch_count")))
    if mismatch is None and agreement is not None and n_value is not None:
        mismatch = int(round((1.0 - float(agreement)) * int(n_value)))

    exact = diag.get("exact_preserved", diag.get("exact_preserved_on_calib", diag.get("fit_exact_preserved")))
    if exact is None and mismatch is not None:
        exact = bool(int(mismatch) == 0)

    return {
        "split": split_label,
        "n": _to_jsonable(n_value),
        "agreement": _to_jsonable(agreement),
        "mismatch_count": _to_jsonable(mismatch),
        "exact_preserved": _to_jsonable(exact),
    }


def _quad_solver_diagnostics(result: Any) -> Dict[str, Any]:
    return _to_jsonable(dict(getattr(result, "solver_diagnostics", None) or {}))


def _build_quad4fhe_run_record(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    fit_diag = _quad_report_diagnostics(result, "fit", n_expected=fit_n)
    test_diag = _quad_report_diagnostics(result, "test", n_expected=test_n)
    solver_diag = _quad_solver_diagnostics(result)

    report_truth_test = _dataframe_test_by_method(getattr(result, "report_vs_truth", None))
    report_pseudo_test = _dataframe_test_by_method(getattr(result, "report_vs_pseudo", None))

    calib_agreement = fit_diag.get("agreement")
    test_agreement = test_diag.get("agreement")
    gap = None
    if calib_agreement is not None and test_agreement is not None:
        gap = float(calib_agreement) - float(test_agreement)

    constraint_version = getattr(result, "constraint_version", None)
    if constraint_version is None:
        constraint_version = solver_diag.get("constraint_version")

    common_solver_keys = [
        "num_pairwise_constraints",
        "min_pairwise_margin",
        "normalized_min_pairwise_margin",
        "slack_positive_count",
        "sum_slack",
        "mean_slack",
        "max_slack",
        "pairwise_slack_positive_count",
        "pairwise_sum_slack",
        "pairwise_mean_slack",
        "pairwise_max_slack",
        "selected_C",
        "soft_C_grid",
        "soft_trace",
        "selected_mu",
        "mu_grid",
        "mu_p",
        "mu_n",
    ]

    quad = {
        "alpha": getattr(result, "alpha", None),
        "beta": getattr(result, "beta", None),
        "eta": getattr(result, "eta", None),
        "threshold": getattr(result, "threshold", None),
        "zero_threshold_realized": getattr(result, "zero_threshold_realized", None),
        "method_used": getattr(result, "method_used", None),
        "hard_feasible": getattr(result, "feasible", None),
        "empirical_margin": getattr(result, "empirical_margin", None),
        "normalized_margin": getattr(result, "normalized_margin", None),
        "quant_decimals": getattr(result, "quant_decimals", None),
        "constraint_version": constraint_version,
        "he_artifact_dir": str(getattr(result, "he_export_dir", None)) if getattr(result, "he_export_dir", None) else None,
        "calib_n": fit_diag.get("n"),
        "calib_agreement": calib_agreement,
        "calib_mismatch_count": fit_diag.get("mismatch_count"),
        "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
        "test_n": test_diag.get("n"),
        "test_agreement": test_agreement,
        "test_mismatch_count": test_diag.get("mismatch_count"),
        "calib_test_agreement_gap": gap,
        "test_top1_acc": _metric_from_table(report_truth_test, "Quad4FHE", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Quad4FHE", "Top5", "Top-5"),
        "test_macro_f1": _metric_from_table(report_truth_test, "Quad4FHE", "MacroF1", "F1"),
        "solver_diagnostics": solver_diag,
    }
    for k in common_solver_keys:
        quad[k] = solver_diag.get(k)

    original = {
        "test_top1_acc": _metric_from_table(report_truth_test, "Original", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Original", "Top5", "Top-5"),
    }

    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "original": original,
        "quad4fhe": quad,
        "report_vs_ground_truth_test": report_truth_test,
        "report_vs_pseudo_labels_test": report_pseudo_test,
        "report_vs_ground_truth_records": _dataframe_records(getattr(result, "report_vs_truth", None)),
        "report_vs_pseudo_labels_records": _dataframe_records(getattr(result, "report_vs_pseudo", None)),
    }
    if extra:
        record.update(_to_jsonable(extra))
    return _to_jsonable(record)


def _collect_autosave_config() -> Dict[str, Any]:
    keys = [
        "SEED", "HIDDEN_DIMS", "HIDDEN_DIM", "EPOCHS", "LR", "BATCH_SIZE", "WEIGHT_DECAY",
        "TRAIN_SIZE", "VAL_SIZE", "TEST_SIZE", "TRAIN_NET_SIZE", "VAL_NET_SIZE",
        "MLP_VAL_FRACTION", "POOL_SIZES_TO_COMPARE", "REPLACEMENT_SELECTION_FRACTION",
        "MIN_POOL_SAMPLES", "MIN_SAMPLES_PER_CLASS", "MIN_PRESENT_CLASSES",
        "NUM_CLASSES", "FIT_SUBSAMPLE_PER_CLASS", "PCA_COMPONENTS", "SVD_COMPONENTS",
        "TFIDF_MAX_FEATURES", "TFIDF_NGRAM_RANGE", "BASELINES", "FLIP_LABELS",
        "USE_HF_DATASETS", "SHUTTLE_SOURCE",
    ]
    g = globals()
    return {k: _to_jsonable(g[k]) for k in keys if k in g}


def _make_autosave_payload() -> Dict[str, Any]:
    g = globals()
    return {
        "dataset": g.get("DATASET_NAME"),
        "experiment": g.get("EXPERIMENT_NAME"),
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_notebook": g.get("SOURCE_NOTEBOOK", None),
        "log_file": str(g.get("LOG_PATH")),
        "seed": g.get("SEED", None),
        "dataset_info": {},
        "config": _collect_autosave_config(),
        "runs": [],
    }


def _ensure_autosave_payload() -> Dict[str, Any]:
    global _AUTOSAVE_PAYLOAD
    if _AUTOSAVE_PAYLOAD is None:
        _AUTOSAVE_PAYLOAD = _make_autosave_payload()
    return _AUTOSAVE_PAYLOAD


def _infer_dataset_info_from_runs(payload: Dict[str, Any]) -> Dict[str, Any]:
    info = dict(payload.get("dataset_info", {}) or {})
    runs = payload.get("runs", [])
    if runs:
        q0 = runs[0].get("quad4fhe", {})
        info.setdefault("calib_n_first", q0.get("calib_n"))
        info.setdefault("test_n", q0.get("test_n"))
        if "NUM_CLASSES" in globals():
            info.setdefault("num_classes", globals().get("NUM_CLASSES"))
        if "PCA_COMPONENTS" in globals():
            info.setdefault("input_dim", globals().get("PCA_COMPONENTS"))
        elif "SVD_COMPONENTS" in globals():
            info.setdefault("input_dim", globals().get("SVD_COMPONENTS"))
    return _to_jsonable(info)


def _summary_row_from_record(record: Dict[str, Any]) -> Dict[str, Any]:
    q = record.get("quad4fhe", {})
    return {
        "key": record.get("key"),
        "hidden_dim": record.get("hidden_dim"),
        "pool_fraction": record.get("pool_fraction"),
        "rep_fit_size": record.get("rep_fit_size"),
        "status": record.get("status", "ok"),
        "method_used": q.get("method_used"),
        "hard_feasible": q.get("hard_feasible"),
        "alpha": q.get("alpha"),
        "beta": q.get("beta"),
        "eta": q.get("eta"),
        "empirical_margin": q.get("empirical_margin"),
        "normalized_margin": q.get("normalized_margin"),
        "quant_decimals": q.get("quant_decimals"),
        "calib_n": q.get("calib_n"),
        "calib_agreement": q.get("calib_agreement"),
        "calib_mismatch_count": q.get("calib_mismatch_count"),
        "exact_preserved_on_calib": q.get("exact_preserved_on_calib"),
        "test_agreement": q.get("test_agreement"),
        "test_mismatch_count": q.get("test_mismatch_count"),
        "calib_test_agreement_gap": q.get("calib_test_agreement_gap"),
        "num_pairwise_constraints": q.get("num_pairwise_constraints"),
        "min_pairwise_margin": q.get("min_pairwise_margin"),
        "slack_positive_count": q.get("slack_positive_count"),
        "sum_slack": q.get("sum_slack"),
        "max_slack": q.get("max_slack"),
        "selected_C": q.get("selected_C"),
        "constraint_version": q.get("constraint_version"),
        "test_top1_acc": q.get("test_top1_acc"),
        "test_top5_acc": q.get("test_top5_acc"),
        "he_artifact_dir": q.get("he_artifact_dir"),
        "skip_reason": record.get("skip_reason"),
    }


def _autosave_write_current_payload() -> None:
    payload = _ensure_autosave_payload()
    payload["dataset_info"] = _infer_dataset_info_from_runs(payload)
    payload["config"] = _collect_autosave_config()
    payload["summary"] = _AUTOSAVE_SUMMARY_ROWS
    if "JSON_PATH" in globals():
        _save_json(payload, globals()["JSON_PATH"])
    if "SUMMARY_CSV_PATH" in globals() and _AUTOSAVE_SUMMARY_ROWS:
        _save_csv(_AUTOSAVE_SUMMARY_ROWS, globals()["SUMMARY_CSV_PATH"])
    _autosave_write_combined_dataset_json()


def _autosave_write_combined_dataset_json() -> None:
    g = globals()
    dataset = g.get("DATASET_NAME")
    if not dataset:
        return
    root = Path("..") / "results" / dataset
    combined = {"dataset": dataset, "created_at": datetime.now().isoformat(timespec="seconds")}
    found = False
    for exp in ("fulltrain", "smallpool"):
        p = root / exp / f"{dataset}_{exp}_results.json"
        if not p.exists():
            continue
        try:
            with p.open("r", encoding="utf-8") as fh:
                block = json.load(fh)
            combined[exp] = {
                "source_json": str(p),
                "source_notebook": block.get("source_notebook"),
                "log_file": block.get("log_file"),
                "dataset_info": block.get("dataset_info", {}),
                "config": block.get("config", {}),
                "runs": block.get("runs", []),
                "summary": block.get("summary", []),
            }
            found = True
        except Exception as exc:
            combined[f"{exp}_read_error"] = str(exc)
    if found:
        _save_json(combined, root / f"{dataset}_results.json")


def _autosave_record_result(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    payload = _ensure_autosave_payload()
    record = _build_quad4fhe_run_record(
        result=result,
        key=key,
        hidden_dim=hidden_dim,
        fit_n=fit_n,
        test_n=test_n,
        pool_fraction=pool_fraction,
        rep_fit_size=rep_fit_size,
        extra=extra,
    )
    payload["runs"].append(record)
    _AUTOSAVE_SUMMARY_ROWS.append(_summary_row_from_record(record))
    _autosave_write_current_payload()
    return record


def _autosave_record_skipped(
    *,
    key: str,
    hidden_dim: Optional[int],
    pool_fraction: Optional[float],
    reason: str,
    rep_fit_size: Optional[int] = None,
) -> Dict[str, Any]:
    payload = _ensure_autosave_payload()
    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "status": "skipped",
        "skip_reason": reason,
        "quad4fhe": {"status": "skipped", "skip_reason": reason},
    }
    payload["runs"].append(_to_jsonable(record))
    _AUTOSAVE_SUMMARY_ROWS.append(_summary_row_from_record(record))
    _autosave_write_current_payload()
    return record


def run_notebook_main_with_autosave(main_fn, log_path):
    global _AUTOSAVE_PAYLOAD, _AUTOSAVE_RUNS, _AUTOSAVE_SUMMARY_ROWS
    _AUTOSAVE_PAYLOAD = None
    _AUTOSAVE_RUNS = []
    _AUTOSAVE_SUMMARY_ROWS = []
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    try:
        with tee_stdout_stderr(log_path):
            print(f"[autosave] dataset={globals().get('DATASET_NAME')} experiment={globals().get('EXPERIMENT_NAME')}")
            print(f"[autosave] output_dir={globals().get('OUTPUT_DIR')}")
            main_fn()
            _autosave_write_current_payload()
    except Exception:
        # Write partial JSON before re-raising, useful for debugging interrupted runs.
        err = traceback.format_exc()
        payload = _ensure_autosave_payload()
        payload["error"] = err
        if "JSON_PATH" in globals():
            _save_json(payload, globals()["JSON_PATH"])
        raise


In [2]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import quad4fhe


# ============================================================
# Autosave configuration
# ============================================================
from pathlib import Path
DATASET_NAME = "SST5"
EXPERIMENT_NAME = "fulltrain"
SOURCE_NOTEBOOK = "SST5_fulltrain_autosave.ipynb"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
SUMMARY_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_summary.csv"

from pathlib import Path


# ============================================================
# Configuration
# ============================================================
SEED = 2026
HIDDEN_DIMS = [64, 128, 256]
EPOCHS = 200
LR = 1e-3
BATCH_SIZE = 512
WEIGHT_DECAY = 1e-5

EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_DELTA = 1e-4

# --- SST-5 data sources (pre-split) ---
SST5_TRAIN_CSV = "sst5_train.csv"
SST5_VAL_CSV   = "sst5_validation.csv"
SST5_TEST_CSV  = "sst5_test.csv"

# --- Text vectorization ---
TFIDF_MAX_FEATURES = 10000
TFIDF_NGRAM_RANGE  = (1, 2)
SVD_COMPONENTS     = 200

NUM_CLASSES = 5
FIT_SUBSAMPLE_PER_CLASS = None

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]


# ============================================================
# Multi-class MLP: Linear -> BN -> ReLU -> Linear
# ============================================================
class MulticlassMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn  = nn.BatchNorm1d(hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.bn(self.fc1(x))))


class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA):
        self.patience, self.min_delta = patience, min_delta
        self.best_loss = float("inf"); self.counter = 0
        self.best_epoch = 0; self.best_state = None

    def step(self, val_loss, epoch, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter, self.best_epoch = val_loss, 0, epoch
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_mlp(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim, num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MulticlassMLP(input_dim, hidden_dim, num_classes).to(device)

    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
    y_va_t = torch.tensor(y_va, dtype=torch.long).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    stopper = EarlyStopping()

    print(f"  Architecture: Linear({input_dim}->{hidden_dim}) -> BN({hidden_dim}) "
          f"-> ReLU -> Linear({hidden_dim}->{num_classes})")
    print(f"  Device={device}, epochs={EPOCHS}, lr={LR}, wd={WEIGHT_DECAY}, batch_size={BATCH_SIZE}")

    n = X_tr_t.shape[0]
    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, device=device)
        losses = []
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss = ce(model(X_tr_t[idx]), y_tr_t[idx])
            loss.backward()
            opt.step()
            losses.append(loss.item())
        train_loss = float(np.mean(losses))

        model.eval()
        with torch.no_grad():
            val_loss = ce(model(X_va_t), y_va_t).item()
            val_acc = (model(X_va_t).argmax(1) == y_va_t).float().mean().item()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"    epoch {epoch+1:>4d}  train_loss={train_loss:.6f}  "
                  f"val_loss={val_loss:.6f}  val_acc={val_acc:.4f}")

        if stopper.step(val_loss, epoch + 1, model):
            print(f"  Early stopping @ epoch {epoch+1} "
                  f"(best val_loss={stopper.best_loss:.6f} @ {stopper.best_epoch})")
            stopper.restore(model)
            break
    else:
        stopper.restore(model)
    model.cpu(); model.eval()
    return model


# ============================================================
# SST-5 loader: pre-split CSVs
# ============================================================
SST5_CLASS_NAMES = {
    0: "very_negative", 1: "negative", 2: "neutral",
    3: "positive",      4: "very_positive",
}


def _read_sst5_csv(path):
    """Read one of the SST-5 CSVs. Handles common column-name variations."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"SST-5 CSV not found: {path}")
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    # Identify label column
    lbl_col = next((c for c in ("label", "labels", "class", "target", "sentiment")
                    if c in df.columns), None)
    # Identify text column
    txt_col = next((c for c in ("text", "sentence", "sentences", "review", "content")
                    if c in df.columns), None)

    if lbl_col is None or txt_col is None:
        # Fallback: assume first column is label, last (or second) is text
        cols = list(df.columns)
        if lbl_col is None:
            lbl_col = cols[0]
        if txt_col is None:
            # pick the last string column that's not the label
            for c in reversed(cols):
                if c != lbl_col and df[c].dtype == object:
                    txt_col = c
                    break
        if txt_col is None:
            raise ValueError(f"Could not identify text column in {path}. Columns: {cols}")

    labels = df[lbl_col].values.astype(int)
    texts  = df[txt_col].astype(str).tolist()
    return texts, labels


def load_sst5():
    """Load SST-5 from the three pre-split CSVs. Returns three (texts, labels) tuples."""
    print(f"Loading SST-5 from:")
    print(f"  train = {SST5_TRAIN_CSV}")
    print(f"  val   = {SST5_VAL_CSV}")
    print(f"  test  = {SST5_TEST_CSV}")

    train_texts, y_train = _read_sst5_csv(SST5_TRAIN_CSV)
    val_texts,   y_val   = _read_sst5_csv(SST5_VAL_CSV)
    test_texts,  y_test  = _read_sst5_csv(SST5_TEST_CSV)

    # Check label range (SST-5 should be 0..4)
    all_labels = np.concatenate([y_train, y_val, y_test])
    if all_labels.min() >= 1:
        # If labels are 1..5, shift to 0..4
        y_train = y_train - 1
        y_val   = y_val - 1
        y_test  = y_test - 1
        all_labels = all_labels - 1

    num_classes = int(all_labels.max() + 1)
    print(f"  SST-5 classes ({num_classes}): {SST5_CLASS_NAMES}")
    print(f"  Sample counts: train={len(train_texts)}, val={len(val_texts)}, test={len(test_texts)}")

    for split_name, lbls in [("train", y_train), ("val", y_val), ("test", y_test)]:
        cnt = np.bincount(lbls, minlength=num_classes)
        print(f"    {split_name} class counts: {cnt.tolist()}")

    return (train_texts, y_train), (val_texts, y_val), (test_texts, y_test), num_classes


def vectorize_text(train_texts, val_texts, test_texts):
    """TF-IDF (fit on train) -> TruncatedSVD (fit on train) -> dense matrix."""
    print(f"  TF-IDF: max_features={TFIDF_MAX_FEATURES}, ngram={TFIDF_NGRAM_RANGE}")
    tfidf = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
        stop_words="english",
        sublinear_tf=True,
        lowercase=True,
    )
    Xtr_sparse = tfidf.fit_transform(train_texts)
    Xva_sparse = tfidf.transform(val_texts)
    Xte_sparse = tfidf.transform(test_texts)
    print(f"  TF-IDF shape: train={Xtr_sparse.shape}, nnz/row≈{Xtr_sparse.nnz/Xtr_sparse.shape[0]:.1f}")

    n_comp = min(SVD_COMPONENTS, Xtr_sparse.shape[1] - 1, Xtr_sparse.shape[0] - 1)
    if n_comp < SVD_COMPONENTS:
        print(f"  Note: reducing SVD n_components from {SVD_COMPONENTS} to {n_comp} (limited by data)")
    svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
    Xtr = svd.fit_transform(Xtr_sparse).astype(np.float64)
    Xva = svd.transform(Xva_sparse).astype(np.float64)
    Xte = svd.transform(Xte_sparse).astype(np.float64)
    print(f"  SVD explained variance: {svd.explained_variance_ratio_.sum():.4f}")
    print(f"  Dense feature shape: train={Xtr.shape}")
    return Xtr, Xva, Xte, tfidf, svd


def stratified_subsample(X, y, n_per_class, seed):
    rng = np.random.default_rng(seed)
    keep = []
    for c in np.unique(y):
        idx_c = np.where(y == c)[0]
        if len(idx_c) > n_per_class:
            idx_c = rng.choice(idx_c, size=n_per_class, replace=False)
        keep.append(idx_c)
    keep = np.concatenate(keep)
    rng.shuffle(keep)
    return X[keep], y[keep]


# ============================================================
# Main
# ============================================================
def main():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    print("=" * 78)
    print("Step 1: Load SST-5 dataset (pre-split text)")
    print("=" * 78)
    (train_texts, y_tr), (val_texts, y_va), (test_texts, y_te), num_classes = load_sst5()
    assert num_classes == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, found {num_classes}."

    print("\n" + "=" * 78)
    print("Step 1b: Vectorize (TF-IDF fit on train -> SVD -> StandardScaler)")
    print("=" * 78)
    Xtr_raw, Xva_raw, Xte_raw, tfidf, svd = vectorize_text(train_texts, val_texts, test_texts)

    scaler = StandardScaler().fit(Xtr_raw)
    X_tr = scaler.transform(Xtr_raw).astype(np.float64)
    X_va = scaler.transform(Xva_raw).astype(np.float64)
    X_te = scaler.transform(Xte_raw).astype(np.float64)

    print(f"  Final shape: train={X_tr.shape}, val={X_va.shape}, test={X_te.shape}")

    all_summaries = []

    for hd in HIDDEN_DIMS:
        print("\n" + "#" * 78)
        print(f"#  RUNNING hidden_dim = {hd}")
        print("#" * 78)

        print(f"\n[Step 2] Train ReLU-MLP with BN (hidden_dim={hd})")
        model = train_mlp(X_tr, y_tr, X_va, y_va, X_tr.shape[1], hd, NUM_CLASSES)

        with torch.no_grad():
            orig_test_logits = model(torch.tensor(X_te, dtype=torch.float32)).cpu().numpy()
        orig_test_acc = float(np.mean(orig_test_logits.argmax(axis=1) == y_te))
        print(f"  Original model test top-1 ACC = {orig_test_acc:.4f}")

        X_fit_full = np.vstack([X_tr, X_va])
        y_fit_full = np.concatenate([y_tr, y_va])
        if FIT_SUBSAMPLE_PER_CLASS is not None:
            X_fit, y_fit = stratified_subsample(
                X_fit_full, y_fit_full, FIT_SUBSAMPLE_PER_CLASS, seed=SEED)
            print(f"  fit_data: subsampled to {len(X_fit)} samples")
        else:
            X_fit, y_fit = X_fit_full, y_fit_full
            print(f"  fit_data: using all {len(X_fit)} samples")

        print(f"\n[Step 3] quad4fhe.replace(task='multiclass', ...)")
        result = quad4fhe.replace(
            task              = "multiclass",
            model             = model,
            hidden_layer      = "fc1",
            output_layer      = "fc2",
            activation        = "relu",
            fit_data          = (X_fit, y_fit),
            test_data         = (X_te, y_te),
            baselines         = BASELINES,
            export_he_dir     = f"he_artifacts_sst5_h{hd}",
            use_cutting_plane = False,
            use_osqp_warmstart= False,
            verbose           = True,
            seed              = SEED,
        )

        summary = {
            "hidden_dim":         hd,
            "method_used":        result.method_used,
            "hard_feasible":      result.feasible,
            "alpha":              result.alpha,
            "beta":               result.beta,
            "eta":                result.eta,
            "empirical_margin":   result.empirical_margin,
            "normalized_margin":  result.normalized_margin,
            "quant_decimals":     result.quant_decimals,
        }
        all_summaries.append(summary)

        _autosave_record_result(
            result=result,
            key=f"hidden_dim={hd}",
            hidden_dim=hd,
            fit_n=len(X_fit),
            test_n=len(X_te),
            pool_fraction=None,
            rep_fit_size=None,
            extra={"summary": summary},
        )

        rm = result.replaced_model.eval()
        with torch.no_grad():
            logits_quad = rm(torch.tensor(X_te, dtype=torch.float32)).cpu().numpy()
        preds_quad = logits_quad.argmax(axis=1)
        print(f"\n  Quad4FHE test top-1 accuracy = {np.mean(preds_quad == y_te):.4f}")

    print("\n" + "=" * 78)
    print("Cross-hidden-dim summary")
    print("=" * 78)
    df = pd.DataFrame(all_summaries)
    with pd.option_context("display.float_format", "{:.6f}".format):
        print(df.to_string(index=False))

    print("\n" + "=" * 78)
    print("Done.")
    print("=" * 78)


if __name__ == "__main__":
    run_notebook_main_with_autosave(main, LOG_PATH)


[autosave] stdout/stderr log -> ..\results\SST5\fulltrain\SST5_fulltrain_result.txt
[autosave] dataset=SST5 experiment=fulltrain
[autosave] output_dir=..\results\SST5\fulltrain
Step 1: Load SST-5 dataset (pre-split text)
Loading SST-5 from:
  train = sst5_train.csv
  val   = sst5_validation.csv
  test  = sst5_test.csv
  SST-5 classes (5): {0: 'very_negative', 1: 'negative', 2: 'neutral', 3: 'positive', 4: 'very_positive'}
  Sample counts: train=8544, val=1101, test=2210
    train class counts: [1092, 2218, 1624, 2322, 1288]
    val class counts: [139, 289, 229, 279, 165]
    test class counts: [279, 633, 389, 510, 399]

Step 1b: Vectorize (TF-IDF fit on train -> SVD -> StandardScaler)
  TF-IDF: max_features=10000, ngram=(1, 2)
  TF-IDF shape: train=(8544, 10000), nnz/row≈8.7
  SVD explained variance: 0.1930
  Dense feature shape: train=(8544, 200)
  Final shape: train=(8544, 200), val=(1101, 200), test=(2210, 200)

#######################################################################